In [ ]:
%load_ext autoreload
%autoreload 2
import waffles
import numpy as np
import json
import shutil 
from tqdm import tqdm

from waffles.input_output.hdf5_structured import load_structured_waveformset
from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.BasicWfAna import BasicWfAna
from waffles.data_classes.IPDict import IPDict
from waffles.data_classes.UniqueChannel import UniqueChannel
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.utils.baseline.baseline import SBaseline
from waffles.np02_utils.AutoMap import generate_ChannelMap, dict_uniqch_to_module, dict_module_to_uniqch, ordered_channels_membrane, ordered_modules_membrane, ordered_modules_pmt, strUch, getModuleName
from waffles.np02_utils.PlotUtils import np02_gen_grids, plot_grid, plot_detectors, genhist, fithist, runBasicWfAnaNP02, runBasicWfAnaNP02Updating
from waffles.np02_utils.load_utils import open_processed

In [ ]:
run = 42830
nwaveforms = 200000
dettype = "pmt"
# dettype = "cathode"

datadir = f"/eos/experiment/neutplatform/protodune/experiments/ProtoDUNE-VD/commissioning/"
endpoint = 106 if dettype == "cathode" else 107 if "membrane" else 110
trigger = "self_trigger"

# Way to low... keep scrollng
dletter = dettype.upper()[0] # C or M...
if dletter != "P":
    group1 = [ f"{dletter}{detnum}({chnum})" for detnum in range(1, 3) for chnum in range(1,3) ]
    group2 = [ f"{dletter}{detnum}({chnum})" for detnum in range(3, 5) for chnum in range(1,3) ]
    group3 = [ f"{dletter}{detnum}({chnum})" for detnum in range(5, 7) for chnum in range(1,3) ]
    group4 = [ f"{dletter}{detnum}({chnum})" for detnum in range(7, 9) for chnum in range(1,3) ]
else:
    n = len(ordered_modules_pmt)
    group1 = ordered_modules_pmt[:n//4]
    group2 = ordered_modules_pmt[n//4:n//2]
    group3 = ordered_modules_pmt[n//2:2*(n//3)]
    group4 = ordered_modules_pmt[2*(n//3):]

In [ ]:
wfset = open_processed(run, dettype, datadir, nwaveforms=nwaveforms, mergefiles=False)

In [ ]:
runBasicWfAnaNP02(wfset, onlyoptimal=False, baselinefinish=60, int_ll=64, int_ul=100, amp_ll=64, amp_ul=150, configyaml="")

In [ ]:
argsheat = dict(
    mode="heatmap",
    analysis_label="std",
    adc_range_above_baseline=1600,
    adc_range_below_baseline=-500,
    adc_bins=200,
    time_bins=wfset.points_per_wf//2,
    filtering=0,
    share_y_scale=False,
    share_x_scale=True,
    wfs_per_axes=2000,
    zlog=True,
)
detector=group1
plot_detectors(wfset, detector, **argsheat)

detector=group2
plot_detectors(wfset, detector, **argsheat)

detector=group3
plot_detectors(wfset, detector, **argsheat)

detector=group4
plot_detectors(wfset, detector, **argsheat)

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm
wfch = ChannelWsGrid.clusterize_waveform_set(wfset)
events_per_daq_window = {}
for module, unch in dict_module_to_uniqch.items():
    ep, ch = unch.endpoint, unch.channel
    if ep not in wfch:
        continue
    if ch not in wfch[ep]:
        continue
    print(f"{module}, {ep}, {ch}, ", end="")
    wfs = wfch[ep][ch]
    if ep not in events_per_daq_window:
        events_per_daq_window[ep] = {}
        
    events_per_daq_window[ep][ch] = []
    timestamps = {}
    for wf in wfs.waveforms:
        timestamps[wf.daq_window_timestamp] = timestamps.get(wf.daq_window_timestamp, []) + [wf.timestamp]
    for daq_time_trigger, list_of_triggers in timestamps.items():
        events_per_daq_window[ep][ch] += [len(list_of_triggers)]
        # print(daq_time_trigger, len(list_of_triggers))
    average_events_per_daq_window = np.mean(events_per_daq_window[ep][ch])
    print(f"{average_events_per_daq_window:.2f} triggers per DAQ window")
        

In [ ]:
from plotly import graph_objects as go
def plot_events_per_daq(wfset:WaveformSet, figure:go.Figure, row, col):

    endpoint = wfset.waveforms[0].endpoint
    channel = wfset.waveforms[0].channel
    values =  events_per_daq_window[endpoint][channel]
    bins = np.linspace(0, 1000, 1000)

    figure.add_trace(
        go.Histogram(
            x=values,
            xaxis='x',
            xbins=dict(start=bins[0], end=bins[-1], size=(bins[1] - bins[0])),
            autobinx=False,

        ),
        row=row, col=col
    )
                        
detector=group1
plot_detectors(wfset, detector, plot_function=plot_events_per_daq)

detector=group2
plot_detectors(wfset, detector, plot_function=plot_events_per_daq)

detector=group3
plot_detectors(wfset, detector, plot_function=plot_events_per_daq)

detector=group4
plot_detectors(wfset, detector, plot_function=plot_events_per_daq)